In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string

In [ ]:
fake = pd.read_csv('/content/drive/MyDrive/ML_Datasets/Fake.csv.zip')
true = pd.read_csv('/content/drive/MyDrive/ML_Datasets/True.csv.zip')

In [ ]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
fake['class'] = 0
true['class'] = 1

In [ ]:
data = pd.concat([fake,true], axis = 0)

In [ ]:
data.sample(10)

,title,text,subject,date,class
18619,SETH RICH MURDER Has CHILLING SIMILARITIES To ...,Just two months shy of the one-year anniversar...,left-news,"May 16, 2017",0
3931,WATCH: Bill Maher Hilariously Thanks Trump Fo...,Bill Maher surprisingly thanked Donald Trump o...,News,"November 5, 2016",0
19503,Armed group seeks legitimacy with Tripoli migr...,"ROME/TUNIS (Reuters) - A powerful armed group,...",worldnews,"September 21, 2017",1
18952,Brazil's top prosecutor says committed to 'Car...,BRASILIA (Reuters) - Brazil s new Prosecutor G...,worldnews,"September 26, 2017",1
6606,Factbox: Contenders for senior jobs in Trump's...,(Reuters) - The following people are mentioned...,politicsNews,"December 22, 2016",1
11536,PRESIDENT TRUMP’S SPEECH: “BELIEVE IN YOURSELF...,THE TRANSCRIPT OF PRESIDENT TRUMP S SPEECH TO ...,politics,"Mar 1, 2017",0
14350,Belarus KGB says Ukrainian journalist set up s...,MINSK (Reuters) - Belarus s KGB state security...,worldnews,"November 20, 2017",1
18672,LGBT Community Furious After Catholic School R...,I fear these cancellations may be based on mi...,left-news,"May 8, 2017",0
20121,MEET THE GUY Milwaukee Is Rioting Over…Interes...,He had a gun that he stole during a burglary i...,left-news,"Aug 15, 2016",0
3825,California governor proposes more money to fig...,SAN FRANCISCO (Reuters) - California governor ...,politicsNews,"May 11, 2017",1


In [ ]:
data = data.drop(['title','subject','date'], axis = 1)

In [ ]:
data.reset_index(inplace = True)

In [ ]:
data.drop(['index'], axis = 1, inplace = True)

In [ ]:
data.sample(10)

,text,class
13623,Next stop after BREXIT is the US! Judge Jeanin...,0
30960,CAIRO (Reuters) - Egyptian President Abdel Fat...,1
31264,(Reuters) - When former President Bill Clinton...,1
8813,The infamous conservative Republican financier...,0
27454,(Reuters) - Representative Joaquin Castro has ...,1
27332,WASHINGTON (Reuters) - President Donald Trump ...,1
1340,It s no secret that German Chancellor Angela M...,0
7469,"On Wednesday morning, President Obama nominate...",0
38492,BEIRUT (Reuters) - British Foreign Secretary B...,1
40865,MOSCOW (Reuters) - Moscow condemns North Korea...,1


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
data["text"] = data["text"].apply(clean_text)

In [ ]:
x = data["text"]
y = data["class"]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.25,
                                                random_state = 42)

In [ ]:
vectorizer = TfidfVectorizer()
xtrain_vectorized = vectorizer.fit_transform(xtrain)
xtest_vectorized = vectorizer.transform(xtest)

In [ ]:
lr = LogisticRegression()
lr.fit(xtrain_vectorized, ytrain)

LogisticRegression()

In [ ]:
prediction = lr.predict(xtest_vectorized)
lr.score(xtest_vectorized, ytest)

0.985924276169265

In [ ]:
print(classification_report(ytest, prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [ ]:
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(lr, 'model.pkl')

['model.pkl']

In [ ]:
from google.colab import files

files.download("model.pkl")
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile app.py

import streamlit as st
import joblib
import os
import google.generativeai as genai

vectorizer = joblib.load("vectorizer.pkl")
model = joblib.load("model.pkl")

st.title("Fake News Detector + AI Summary")

st.write("Enter a news article to check whether it is Fake or Real and get AI summary.")

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    genai.configure(api_key=api_key)
    gemini_model = genai.GenerativeModel("gemini-1.5-flash")
else:
    gemini_model = None

news_input = st.text_area("News Article:")

if st.button("Check News"):

    if news_input.strip():

        # ML Prediction
        transformed_input = vectorizer.transform([news_input])
        prediction = model.predict(transformed_input)

        if prediction[0] == 1:
            st.success("The News is Real!")
        else:
            st.error("The News is Fake!")

        if gemini_model:
            with st.spinner("Generating AI summary..."):
                response = gemini_model.generate_content(
                    "Summarize the following news article in 4-5 bullet points:\n\n"
                    + news_input
                )

            st.subheader("News Summary")
            st.write(response.text)

        else:
            st.warning("Gemini API key not configured.")

    else:
        st.warning("Please enter a news article.")

Overwriting app.py


In [ ]:
%%writefile requirements.txt
streamlit
joblib
scikit-learn
pandas
numpy
google-generativeai

Overwriting requirements.txt


In [ ]:
!pip install streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx localtunnel --port 8501

⠙⠹your url is: https://loud-knives-rule.loca.lt
